# Section 1 — Load Existing Data

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OneHotEncoder

current_directory = os.path.abspath('')
if os.path.exists(os.path.join(current_directory, 'Phase_1_Machine_Learning')):
    root_directory = current_directory
elif os.path.exists(os.path.abspath(os.path.join(current_directory, '..', 'Phase_1_Machine_Learning'))):
    root_directory = os.path.abspath(os.path.join(current_directory, '..'))
elif os.path.exists(os.path.abspath(os.path.join(current_directory, '..', '..', 'Phase_1_Machine_Learning'))):
    root_directory = os.path.abspath(os.path.join(current_directory, '..', '..'))
else:
    root_directory = current_directory

data_directory = os.path.join(root_directory, 'Phase_1_Machine_Learning', 'data', 'GeneratedOutput', '02.Preprocessing&FeatureSelection')

train_df = pd.read_parquet(os.path.join(data_directory, 'train.parquet'))
val_df = pd.read_parquet(os.path.join(data_directory, 'val.parquet'))
test_df = pd.read_parquet(os.path.join(data_directory, 'test.parquet'))

target_column = 'ccme_values'
target_quarantine_columns = ['ccme_values', 'ccme_wqi', 'ccme_wqi_code']

print(f'Training dataset shape: {train_df.shape}')
print(f'Validation dataset shape: {val_df.shape}')
print(f'Test dataset shape: {test_df.shape}')
print(f'Target column verified: {target_column in train_df.columns}')
for col in target_quarantine_columns:
    print(f'Target column present in raw partitions: {col} -> {col in train_df.columns}')

Training dataset shape: (1973731, 45)
Validation dataset shape: (422943, 45)
Test dataset shape: (422943, 45)
Target column verified: True
Target column present in raw partitions: ccme_values -> True
Target column present in raw partitions: ccme_wqi -> True
Target column present in raw partitions: ccme_wqi_code -> True


# Section 2 — Verify Existing Features

In [2]:
metadata_file_path = os.path.join(data_directory, 'preprocessing_metadata.json')
with open(metadata_file_path, 'r') as file_handle:
    metadata = json.load(file_handle)

input_features = metadata['input_features_ordered']
categorical_features = metadata['context_features']
numerical_features = [col for col in input_features if col not in categorical_features]
historical_features = metadata['historical_features']

print(f'Total features in Phase 1 metadata: {len(input_features)}')
print(f'Numerical features count: {len(numerical_features)}')
print(f'Categorical features count: {len(categorical_features)}')
print(f'Historical features count: {len(historical_features)}')

assert len(input_features) == 38
assert len(numerical_features) == 36
assert len(categorical_features) == 2
assert len(historical_features) == 6
assert all(col in train_df.columns for col in input_features)
assert all(col in val_df.columns for col in input_features)
assert all(col in test_df.columns for col in input_features)

Total features in Phase 1 metadata: 38
Numerical features count: 36
Categorical features count: 2
Historical features count: 6


# Section 3 — Historical Feature Validity and Leakage Investigation

In [3]:
for feature in historical_features:
    train_nulls = int(train_df[feature].isna().sum())
    val_nulls = int(val_df[feature].isna().sum())
    test_nulls = int(test_df[feature].isna().sum())
    assert train_nulls == 0
    assert val_nulls == 0
    assert test_nulls == 0
    print(f'{feature:25s} | min: {train_df[feature].min():.4f} | mean: {train_df[feature].mean():.4f} | max: {train_df[feature].max():.4f}')

correlations = {}
for feature in input_features:
    correlations[feature] = float(train_df[feature].corr(train_df[target_column]))

sorted_correlations = sorted(correlations.items(), key=lambda item: abs(item[1]), reverse=True)

print('Top absolute correlations with ccme_values in training set:')
for feature_name, correlation_value in sorted_correlations[:10]:
    print(f'{feature_name:30s}: r = {correlation_value:.4f}')

for target in target_quarantine_columns:
    assert target not in input_features

hist_do_lag1              | min: 0.0000 | mean: 10.0179 | max: 20.0000
hist_temp_lag1            | min: 0.0000 | mean: 11.7058 | max: 45.0000
hist_bod_lag1             | min: 0.0000 | mean: 4.7761 | max: 255.0000
hist_ortho_lag1           | min: 0.0000 | mean: 0.7001 | max: 100.0000
hist_do_rolling_3         | min: 0.0000 | mean: 10.0236 | max: 19.8000
hist_temp_rolling_3       | min: 0.0000 | mean: 11.6818 | max: 45.0000
Top absolute correlations with ccme_values in training set:
log1p_orthophosphate          : r = -0.8533
exceedance_counter            : r = -0.7778
orthophosphate_mg_l           : r = -0.6605
log1p_ammonia                 : r = -0.6596
nitrogen_mg_l                 : r = -0.6111
log1p_nitrogen                : r = -0.5843
log1p_bod                     : r = -0.5776
hist_ortho_lag1               : r = -0.5364
log1p_np_ratio                : r = 0.5170
log1p_bod_do_ratio            : r = -0.4570


# Section 4 — Numerical Feature Preparation

In [4]:
print(f'Numerical features count: {len(numerical_features)}')
print('Numerical feature data types:')
print(train_df[numerical_features].dtypes.value_counts())

scaler = StandardScaler()
x_train_num = scaler.fit_transform(train_df[numerical_features].astype(np.float32))
x_val_num = scaler.transform(val_df[numerical_features].astype(np.float32))
x_test_num = scaler.transform(test_df[numerical_features].astype(np.float32))

train_num_mean_max_deviation = float(np.abs(x_train_num.mean(axis=0)).max())
train_num_std_max_deviation = float(np.abs(x_train_num.std(axis=0) - 1.0).max())

print(f'Maximum absolute mean deviation from 0.0: {train_num_mean_max_deviation:.6e}')
print(f'Maximum absolute std deviation from 1.0: {train_num_std_max_deviation:.6e}')
print(f'NaN count in transformed train numerical: {np.isnan(x_train_num).sum()}')
print(f'NaN count in transformed val numerical: {np.isnan(x_val_num).sum()}')
print(f'NaN count in transformed test numerical: {np.isnan(x_test_num).sum()}')
print(f'Infinite count in transformed train numerical: {np.isinf(x_train_num).sum()}')
print(f'Infinite count in transformed val numerical: {np.isinf(x_val_num).sum()}')
print(f'Infinite count in transformed test numerical: {np.isinf(x_test_num).sum()}')

Numerical features count: 36
Numerical feature data types:
float64    32
int64       4
Name: count, dtype: int64
Maximum absolute mean deviation from 0.0: 5.599672e-07
Maximum absolute std deviation from 1.0: 5.960464e-08
NaN count in transformed train numerical: 0
NaN count in transformed val numerical: 0
NaN count in transformed test numerical: 0
Infinite count in transformed train numerical: 0
Infinite count in transformed val numerical: 0
Infinite count in transformed test numerical: 0


# Section 5 — Categorical Feature Preparation

In [5]:
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
x_train_cat = categorical_encoder.fit_transform(train_df[categorical_features])
x_val_cat = categorical_encoder.transform(val_df[categorical_features])
x_test_cat = categorical_encoder.transform(test_df[categorical_features])

one_hot_feature_names = list(categorical_encoder.get_feature_names_out(categorical_features))
print(f'One-hot encoded categories count: {len(one_hot_feature_names)}')
print(f'One-hot feature names: {one_hot_feature_names}')

x_train_combined = np.hstack([x_train_num, x_train_cat]).astype(np.float32)
x_val_combined = np.hstack([x_val_num, x_val_cat]).astype(np.float32)
x_test_combined = np.hstack([x_test_num, x_test_cat]).astype(np.float32)

all_feature_names = numerical_features + one_hot_feature_names
print(f'Combined input features count: {len(all_feature_names)}')
print(f'Train combined shape: {x_train_combined.shape}')
print(f'Val combined shape: {x_val_combined.shape}')
print(f'Test combined shape: {x_test_combined.shape}')

One-hot encoded categories count: 17
One-hot feature names: ['country_code_0', 'country_code_1', 'country_code_2', 'country_code_3', 'country_code_4', 'waterbody_type_code_0', 'waterbody_type_code_1', 'waterbody_type_code_2', 'waterbody_type_code_3', 'waterbody_type_code_4', 'waterbody_type_code_5', 'waterbody_type_code_6', 'waterbody_type_code_7', 'waterbody_type_code_8', 'waterbody_type_code_9', 'waterbody_type_code_10', 'waterbody_type_code_11']
Combined input features count: 53
Train combined shape: (1973731, 53)
Val combined shape: (422943, 53)
Test combined shape: (422943, 53)


# Section 6 — PyTorch Dataset and DataLoader Preparation

In [6]:
y_train = train_df[target_column].values.astype(np.float32).reshape(-1, 1)
y_val = val_df[target_column].values.astype(np.float32).reshape(-1, 1)
y_test = test_df[target_column].values.astype(np.float32).reshape(-1, 1)

x_train_num_tensor = torch.from_numpy(x_train_num).float()
x_train_cat_tensor = torch.from_numpy(x_train_cat).float()
x_train_all_tensor = torch.from_numpy(x_train_combined).float()
y_train_tensor = torch.from_numpy(y_train).float()

x_val_num_tensor = torch.from_numpy(x_val_num).float()
x_val_cat_tensor = torch.from_numpy(x_val_cat).float()
x_val_all_tensor = torch.from_numpy(x_val_combined).float()
y_val_tensor = torch.from_numpy(y_val).float()

x_test_num_tensor = torch.from_numpy(x_test_num).float()
x_test_cat_tensor = torch.from_numpy(x_test_cat).float()
x_test_all_tensor = torch.from_numpy(x_test_combined).float()
y_test_tensor = torch.from_numpy(y_test).float()

class WaterQualityDataset(Dataset):
    def __init__(self, numerical_tensor, categorical_tensor, features_tensor, target_tensor, return_mode='combined'):
        self.numerical_tensor = numerical_tensor
        self.categorical_tensor = categorical_tensor
        self.features_tensor = features_tensor
        self.target_tensor = target_tensor
        self.return_mode = return_mode

    def __len__(self):
        return len(self.target_tensor)

    def __getitem__(self, index):
        if self.return_mode == 'separated':
            return self.numerical_tensor[index], self.categorical_tensor[index], self.target_tensor[index]
        return self.features_tensor[index], self.target_tensor[index]

train_dataset = WaterQualityDataset(x_train_num_tensor, x_train_cat_tensor, x_train_all_tensor, y_train_tensor)
val_dataset = WaterQualityDataset(x_val_num_tensor, x_val_cat_tensor, x_val_all_tensor, y_val_tensor)
test_dataset = WaterQualityDataset(x_test_num_tensor, x_test_cat_tensor, x_test_all_tensor, y_test_tensor)

batch_size = 1024
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Training dataset length: {len(train_dataset)}')
print(f'Validation dataset length: {len(val_dataset)}')
print(f'Test dataset length: {len(test_dataset)}')
print(f'Training DataLoader batches: {len(train_loader)}')
print(f'Validation DataLoader batches: {len(val_loader)}')
print(f'Test DataLoader batches: {len(test_loader)}')

Training dataset length: 1973731
Validation dataset length: 422943
Test dataset length: 422943
Training DataLoader batches: 1928
Validation DataLoader batches: 414
Test DataLoader batches: 414


# Section 7 — Standardized Interface for Team Members

In [7]:
interface_summary = {
    'input_dimension_total': len(all_feature_names),
    'numerical_dimension': len(numerical_features),
    'categorical_dimension': len(one_hot_feature_names),
    'batch_size': batch_size,
    'train_batches': len(train_loader),
    'val_batches': len(val_loader),
    'test_batches': len(test_loader),
    'target_column': target_column,
}

print('STANDARDIZED INTERFACE SPECIFICATION FOR TEAM MEMBERS:')
for key, val in interface_summary.items():
    print(f'  {key:25s}: {val}')

sample_combined_batch, sample_target_batch = next(iter(train_loader))
print(f'Default DataLoader output: batch_x shape = {tuple(sample_combined_batch.shape)}, batch_y shape = {tuple(sample_target_batch.shape)}')

separated_train_dataset = WaterQualityDataset(x_train_num_tensor, x_train_cat_tensor, x_train_all_tensor, y_train_tensor, return_mode='separated')
separated_train_loader = DataLoader(separated_train_dataset, batch_size=batch_size, shuffle=False)
sample_num, sample_cat, sample_y = next(iter(separated_train_loader))
print(f'Separated DataLoader output: batch_num = {tuple(sample_num.shape)}, batch_cat = {tuple(sample_cat.shape)}, batch_y = {tuple(sample_y.shape)}')

STANDARDIZED INTERFACE SPECIFICATION FOR TEAM MEMBERS:
  input_dimension_total    : 53
  numerical_dimension      : 36
  categorical_dimension    : 17
  batch_size               : 1024
  train_batches            : 1928
  val_batches              : 414
  test_batches             : 414
  target_column            : ccme_values
Default DataLoader output: batch_x shape = (1024, 53), batch_y shape = (1024, 1)
Separated DataLoader output: batch_num = (1024, 36), batch_cat = (1024, 17), batch_y = (1024, 1)


# Section 8 — Save Phase 2 Artifacts

In [8]:
phase2_artifacts_directory = os.path.join(root_directory, 'Phase_2_Deep_Learning', 'artifacts')
phase2_configs_directory = os.path.join(root_directory, 'Phase_2_Deep_Learning', 'configs')
phase2_processed_directory = os.path.join(root_directory, 'Phase_2_Deep_Learning', 'data', 'processed')

os.makedirs(phase2_artifacts_directory, exist_ok=True)
os.makedirs(phase2_configs_directory, exist_ok=True)
os.makedirs(phase2_processed_directory, exist_ok=True)

scaler_save_path = os.path.join(phase2_artifacts_directory, 'numerical_scaler.joblib')
joblib.dump(scaler, scaler_save_path)

encoder_save_path = os.path.join(phase2_artifacts_directory, 'categorical_encoder.joblib')
joblib.dump(categorical_encoder, encoder_save_path)

feature_config = {
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'one_hot_feature_names': one_hot_feature_names,
    'all_feature_names': all_feature_names,
    'num_numerical_features': len(numerical_features),
    'num_categorical_features': len(categorical_features),
    'num_one_hot_features': len(one_hot_feature_names),
    'total_input_dimension': len(all_feature_names),
    'target_variable': target_column,
    'batch_size': batch_size,
    'train_rows': len(train_dataset),
    'val_rows': len(val_dataset),
    'test_rows': len(test_dataset),
}

config_save_path = os.path.join(phase2_artifacts_directory, 'feature_config.json')
with open(config_save_path, 'w') as file_handle:
    json.dump(feature_config, file_handle, indent=2)

with open(os.path.join(phase2_configs_directory, 'feature_config.json'), 'w') as file_handle:
    json.dump(feature_config, file_handle, indent=2)

sample_features, sample_targets = next(iter(train_loader))
sample_batch_payload = {
    'sample_features': sample_features[:32],
    'sample_targets': sample_targets[:32],
    'feature_names': all_feature_names,
    'input_dimension': len(all_feature_names),
}
torch.save(sample_batch_payload, os.path.join(phase2_artifacts_directory, 'sample_batch.pt'))
torch.save(sample_batch_payload, os.path.join(phase2_processed_directory, 'sample_batch.pt'))

print(f'Scaler saved: {os.path.exists(scaler_save_path)}')
print(f'Encoder saved: {os.path.exists(encoder_save_path)}')
print(f'Config saved: {os.path.exists(config_save_path)}')
print(f'Sample batch saved: {os.path.exists(os.path.join(phase2_artifacts_directory, "sample_batch.pt"))}')

Scaler saved: True
Encoder saved: True
Config saved: True
Sample batch saved: True


# Section 9 — Final Verification

In [9]:
batch_features, batch_targets = next(iter(train_loader))
val_batch_features, val_batch_targets = next(iter(val_loader))
test_batch_features, test_batch_targets = next(iter(test_loader))

assert batch_features.shape == (batch_size, len(all_feature_names))
assert batch_targets.shape == (batch_size, 1)
assert not torch.isnan(batch_features).any()
assert not torch.isinf(batch_features).any()
assert not torch.isnan(batch_targets).any()
assert not torch.isnan(val_batch_features).any()
assert not torch.isnan(test_batch_features).any()

print('=' * 60)
print('FINAL PIPELINE VERIFICATION SUMMARY')
print('=' * 60)
print(f'Number of training observations:   {len(train_dataset):,}')
print(f'Number of validation observations: {len(val_dataset):,}')
print(f'Number of test observations:       {len(test_dataset):,}')
print(f'Number of numerical features:      {len(numerical_features)}')
print(f'Number of categorical features:    {len(categorical_features)}')
print(f'Number of one-hot encoded columns: {len(one_hot_feature_names)}')
print(f'Final input dimensions:            {len(all_feature_names)}')
print(f'Batch input shape:                 {tuple(batch_features.shape)}')
print(f'Batch target shape:                {tuple(batch_targets.shape)}')
print(f'Target variable:                   {target_column}')
print('Verification passed successfully.')
print('=' * 60)

FINAL PIPELINE VERIFICATION SUMMARY
Number of training observations:   1,973,731
Number of validation observations: 422,943
Number of test observations:       422,943
Number of numerical features:      36
Number of categorical features:    2
Number of one-hot encoded columns: 17
Final input dimensions:            53
Batch input shape:                 (1024, 53)
Batch target shape:                (1024, 1)
Target variable:                   ccme_values
Verification passed successfully.
